In [5]:
import pandas as pd
import re
from difflib import SequenceMatcher

# =========================
# Paths
# =========================
ieee_file = "IEEE Xplore Citation Plain Text Download 2026.4.21.12.4.35.txt"
scopus_file = "scopus_export_Apr 21-2026_30f8c46d-ecc2-4826-ab18-5c589e340d92.txt"
output_file = "articles_scopus_ieee.xlsx"

# =========================
# Helpers
# =========================
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text

def similarity(a, b):
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

def safe_strip(x):
    return x.strip() if isinstance(x, str) else x

# =========================
# IEEE parser
# =========================
def parse_ieee_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # records usually start with author list + quoted title and end before next blank line block
    # split on blank lines followed by something that looks like a new citation
    chunks = re.split(r'\n\s*\n(?=[A-Z][^"\n]+,\s*")', text.strip(), flags=re.MULTILINE)

    rows = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        lines = [ln.strip() for ln in chunk.splitlines() if ln.strip()]
        first_line = lines[0] if lines else ""

        # Title between quotes on first line
        m_title = re.search(r'"([^"]+)"', first_line)
        title = m_title.group(1).strip() if m_title else None

        # Authors = text before first quoted title
        authors = None
        if m_title:
            authors = first_line[:m_title.start()].rstrip(" ,")

        # DOI may appear in first line as doi: ...
        m_doi = re.search(r'doi:\s*([^\s,]+)', chunk, flags=re.IGNORECASE)
        doi = m_doi.group(1).strip() if m_doi else None

        # Year: try 4-digit year in first line
        m_year = re.search(r'\b(20\d{2}|19\d{2})\b', first_line)
        year = m_year.group(1) if m_year else None

        # Abstract
        m_abs = re.search(r'Abstract:\s*(.*?)(?=\nkeywords:|\nURL:|$)', chunk, flags=re.IGNORECASE | re.DOTALL)
        abstract = m_abs.group(1).strip() if m_abs else None

        # Keywords
        m_kw = re.search(r'keywords:\s*\{(.*?)\}', chunk, flags=re.IGNORECASE | re.DOTALL)
        keywords = m_kw.group(1).strip() if m_kw else None

        # URL
        m_url = re.search(r'URL:\s*(\S+)', chunk, flags=re.IGNORECASE)
        url = m_url.group(1).strip() if m_url else None

        if title:
            rows.append({
                "source": "IEEE",
                "title": safe_strip(title),
                "authors": safe_strip(authors),
                "abstract": safe_strip(abstract),
                "year": safe_strip(year),
                "doi": safe_strip(doi),
                "keywords": safe_strip(keywords),
                "url": safe_strip(url),
                "documentType": None
            })

    return pd.DataFrame(rows)

# =========================
# Scopus parser
# =========================
def parse_scopus_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # remove export header
    text = re.sub(r'^Scopus\s+EXPORT DATE:.*?\n+', '', text, flags=re.IGNORECASE | re.DOTALL)

    # each record in your file starts with author line, then AUTHOR FULL NAMES:
    chunks = re.split(r'\n\s*\n(?=[^\n]+\nAUTHOR FULL NAMES:)', text.strip(), flags=re.MULTILINE)

    rows = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        lines = [ln.strip() for ln in chunk.splitlines() if ln.strip()]
        if len(lines) < 4:
            continue

        # record pattern from your file:
        # line0: short authors
        # line1: AUTHOR FULL NAMES: ...
        # line2: ids
        # line3: title
        authors = lines[0]
        title = lines[3] if len(lines) > 3 else None

        # DOI
        m_doi = re.search(r'DOI:\s*([^\s]+)', chunk, flags=re.IGNORECASE)
        doi = m_doi.group(1).strip() if m_doi else None

        # Year from source line like "(2023) ICASSP ..."
        m_year = re.search(r'\((20\d{2}|19\d{2})\)', chunk)
        year = m_year.group(1) if m_year else None

        # URL
        m_url = re.search(r'https?://\S+', chunk)
        url = m_url.group(0).strip() if m_url else None

        # Abstract
        m_abs = re.search(
            r'ABSTRACT:\s*(.*?)(?=\nDOCUMENT TYPE:|\nPUBLICATION STAGE:|\nOPEN ACCESS:|\nSOURCE:|$)',
            chunk,
            flags=re.IGNORECASE | re.DOTALL
        )
        abstract = m_abs.group(1).strip() if m_abs else None

        # Document type
        m_doc = re.search(r'DOCUMENT TYPE:\s*(.*)', chunk, flags=re.IGNORECASE)
        document_type = m_doc.group(1).strip() if m_doc else None

        if title:
            rows.append({
                "source": "Scopus",
                "title": safe_strip(title),
                "authors": safe_strip(authors),
                "abstract": safe_strip(abstract),
                "year": safe_strip(year),
                "doi": safe_strip(doi),
                "keywords": None,
                "url": safe_strip(url),
                "documentType": safe_strip(document_type)
            })

    return pd.DataFrame(rows)

# =========================
# Load and merge
# =========================
df_ieee = parse_ieee_txt(ieee_file)
df_scopus = parse_scopus_txt(scopus_file)

print("IEEE parsed:", len(df_ieee))
print("Scopus parsed:", len(df_scopus))

df = pd.concat([df_scopus, df_ieee], ignore_index=True)

# keep only rows with titles
df = df[df["title"].notna() & (df["title"].astype(str).str.len() > 3)].copy()
df.reset_index(drop=True, inplace=True)

# =========================
# Duplicate detection
# =========================
df["isDuplicate"] = False
df["similarityScore"] = 0.0
df["bestMatchRow"] = None
df["duplicateReason"] = None

for i in range(len(df)):
    title_i = df.loc[i, "title"]
    doi_i = str(df.loc[i, "doi"]).strip().lower() if pd.notna(df.loc[i, "doi"]) else ""

    for j in range(i):
        title_j = df.loc[j, "title"]
        doi_j = str(df.loc[j, "doi"]).strip().lower() if pd.notna(df.loc[j, "doi"]) else ""

        # Exact DOI match
        if doi_i and doi_j and doi_i == doi_j:
            df.loc[i, "isDuplicate"] = True
            df.loc[i, "similarityScore"] = 1.0
            df.loc[i, "bestMatchRow"] = j
            df.loc[i, "duplicateReason"] = "same DOI"
            break

        # Title similarity
        score = similarity(title_i, title_j)

        if score > df.loc[i, "similarityScore"]:
            df.loc[i, "similarityScore"] = round(score, 4)
            df.loc[i, "bestMatchRow"] = j

        if score >= 0.92:
            df.loc[i, "isDuplicate"] = True
            df.loc[i, "duplicateReason"] = "similar title"
            break

# =========================
# Save to Excel
# =========================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="records", index=False)

    counts = pd.DataFrame({
        "metric": [
            "IEEE parsed",
            "Scopus parsed",
            "Merged total",
            "Duplicates flagged",
            "Unique remaining (not removed)"
        ],
        "value": [
            len(df_ieee),
            len(df_scopus),
            len(df),
            int(df["isDuplicate"].sum()),
            int((~df["isDuplicate"]).sum())
        ]
    })
    counts.to_excel(writer, sheet_name="counts", index=False)

print(f"\nDone: {output_file}")
print("Merged total:", len(df))
print("Duplicates flagged:", int(df['isDuplicate'].sum()))
display(df.head(10))
print("\nSTATISTICS")
print("Total articles:", len(df))
print("Duplicates:", df["isDuplicate"].sum())
print("Unique articles:", len(df) - df["isDuplicate"].sum())

IEEE parsed: 15
Scopus parsed: 124

Done: articles_scopus_ieee.xlsx
Merged total: 139
Duplicates flagged: 5


,source,title,authors,abstract,year,doi,keywords,url,documentType,isDuplicate,similarityScore,bestMatchRow,duplicateReason
0,Scopus,Vision2Touch: Imaging Estimation of Surface Ta...,"Chen J., Zhou S.",Similar to the human's multiple perception sys...,2023,10.1109/ICASSP49357.2023.10097053,None,https://www.scopus.com/pages/publications/8600...,Conference paper,False,0.0000,None,None
1,Scopus,TIRgel: A Visuo-Tactile Sensor With Total Inte...,"Zhang S., Sun Y., Shan J., Chen Z., Sun F., Ya...",This letter proposes a vision-based tactile se...,2023,10.1109/LRA.2023.3306670,None,https://www.scopus.com/pages/publications/8516...,Article,False,0.2581,0,None
2,Scopus,Estimating indoor tile friction coefficient us...,"Yang J.H., Yoon K.-I., Ha S., Hong A., Lim S.-C.",Slip and fall accidents are common both indoor...,2025,10.1093/jcde/qwaf003,None,https://www.scopus.com/pages/publications/8521...,Article,False,0.3913,0,None
3,Scopus,Visually Indicated Sounds,"Owens A., Isola P., McDermott J., Torralba A.,...",Objects make distinctive sounds when they are ...,2016,10.1109/CVPR.2016.264,None,https://www.scopus.com/pages/publications/8498...,Conference paper,False,0.2796,2,None
4,Scopus,Artificial morality basic device: transistor f...,"Chen S., Yu R., Zou Y., Yu X., Liu C., Hu Y., ...",The extensive application of increasingly soph...,2024,10.1007/s40843-023-2710-0,None,https://www.scopus.com/pages/publications/8518...,Article,False,0.3158,1,None
5,Scopus,Re-Evaluating Parallel Finger-Tip Tactile Sens...,"Zhang F., Corke P.",Finger-tip tactile sensors are increasingly us...,2023,10.1109/IROS55552.2023.10342262,None,https://www.scopus.com/pages/publications/8518...,Conference paper,False,0.3925,1,None
6,Scopus,Surface Material Recognition Using Active Mult...,"Liu H., Fang J., Xu X., Sun F.","Visual, haptic, and auditory modalities can pr...",2018,10.1007/s12559-018-9571-z,None,https://www.scopus.com/pages/publications/8504...,Article,False,0.4138,2,None
7,Scopus,Quality-aware massive content delivery in digi...,"Gao Y., Liao J., Wei X., Zhou L.",Massive content delivery will become one of th...,2023,10.23919/JCC.2023.02.001,None,https://www.scopus.com/pages/publications/8515...,Article,False,0.3232,3,None
8,Scopus,Nonvolatile memory and neuromorphic devices ba...,"Zhang Y., Wang L., Huang Z., Deng W., Yan X., ...","Two-dimensional (2D) materials, owing to their...",2025,10.1016/j.apmt.2025.102908,None,https://www.scopus.com/pages/publications/1050...,Article,False,0.3483,4,None
9,Scopus,Sensor2Text: Enabling Natural Language Interac...,"Chen W., Cheng J., Wang L., Zhao W., Matusik W.","Visual Question-Answering, a technology that g...",2024,10.1145/3699747,None,https://www.scopus.com/pages/publications/8521...,Article,False,0.4260,2,None



STATISTICS
Total articles: 139
Duplicates: 5
Unique articles: 134
